In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt

In [ ]:
# all this to import the KeplerClocks code
from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks

In [ ]:
kicid = "KIC009899416"
clock_table = clocks.best_clocks_in_star(kicid)
print(clock_table)

In [ ]:
def jackknife_timing_variance(row, njack=8):
    # unpack
    kicid, om, M, value = row['kicid'], row['angular_frequency'], row['fourier_series_degree'], row['theoretical_value']

    # get data
    ts, ys, errs, deltaf, deltat = clocks.get_kepler_data(kicid)
    ivars = 1. / errs ** 2
    X, ms, pars = clocks.fourier_wls_fit(om, M, ts, ys, ivars)
    resids = ys - X @ pars
    deriv_pars = clocks.take_derivative_wrt_phase(pars, ms)
    derivs = om * X @ deriv_pars # dflux / dt

    # start jackknife
    subsamp = np.arange(len(ys)) * njack // len(ys)
    time_shifts = np.zeros(njack) + np.nan
    for samp in range(njack):
        loo = subsamp != samp
        time_shifts[samp] = np.sum(ivars[loo] * resids[loo] * derivs[loo]) \
                          / np.sum(ivars[loo] * derivs[loo] * derivs[loo])
    return ((njack - 1) / njack) * np.sum((time_shifts - np.mean(time_shifts)) ** 2)

In [ ]:
for row in clock_table:
    jack_var = jackknife_timing_variance(row)
    jack_value = 1. / jack_var
    print(f"{row['kicid']} {row['angular_frequency']:.6f} {row['theoretical_value']:.2e} {jack_value:.2e}")